# Intro to flock

Small examples with plain Python values.

## 1. Hello from each rank

In [1]:
import flock


@flock.distribute(workers=4)
async def hello():
    return f"Hello World of {flock.get_world_size()} ranks, I'm {flock.get_rank()}"


hello()

["Hello World of 4 ranks, I'm 0",
 "Hello World of 4 ranks, I'm 1",
 "Hello World of 4 ranks, I'm 2",
 "Hello World of 4 ranks, I'm 3"]

## 2. Send one message

In [2]:
@flock.distribute(workers=2)
async def ping():
    rank = flock.get_rank()

    if rank == 0:
        await flock.send(1, "ping")
        return f"{rank} sent message"
    else:
        msg = await flock.recv(0)
        return f"{rank} received '{msg}'"


ping()

['0 sent message', "1 received 'ping'"]

## 3. Ring communication

In [3]:
@flock.distribute(workers=4)
async def ring():
    rank = flock.get_rank()
    world = flock.get_world_size()

    dst = (rank + 1) % world
    src = (rank - 1) % world

    await flock.isend(dst, f"message from {rank} to {dst}").wait()
    return await flock.recv(src)


ring()

['message from 3 to 0',
 'message from 0 to 1',
 'message from 1 to 2',
 'message from 2 to 3']

## 4. Post early, wait later

In [4]:
@flock.distribute(workers=2)
async def overlap_shape():
    rank = flock.get_rank()

    if rank == 0:
        work = flock.isend(1, "data")

        for _ in range(10):
            pass
        local_result = "rank 0 did local work"

        await work.wait()
        return local_result
    else:
        work = flock.irecv(0)
        
        for _ in range(10):
            pass
        local_result = "rank 1 did local work"

        msg = await work.wait()
        return f"{local_result}, then received {msg}"


overlap_shape()

['rank 0 did local work', 'rank 1 did local work, then received data']